# COde 00 — Research wheel bank v5

Chạy ONLINE trên Linux/Kaggle có cùng Python/scientific stack với phiên GPU đích.
Nếu RTX PRO 6000 chặn Internet: notebook00 dùng Accelerator=None (CPU), Internet ON.
Notebook01/02 mới dùng Pro6000, Internet OFF. Lỗi đọc Secrets chưa chứng minh HF token sai.
TARGET_ENVIRONMENT có thể giữ inventory từ MODE='inspect' trên GPU để so trước khi tải.
Gắn source 0.5.1 đã giải nén. Chưa có prepared release vẫn chạy: RUN_MINI_REHEARSAL=False.
Bank chứa project wheel, dependency wheelhouse + requirements.lock, source/configs và pretrained weights có checksum.
Chấp nhận điều khoản DINOv3-B/L, thêm Kaggle Secret HF_TOKEN và cấp quyền cho notebook.
Hướng dẫn: _support/KAGGLE_RUNBOOK.md. Bank v4 không dùng cho source mới.


In [ ]:
from pathlib import Path
import os, sys, json, importlib.util
TARGET_ENVIRONMENT = None  # paste inventory from Pro6000 inspect; CPU stack must match
MODE = 'prepare'  # inspect để xem môi trường GPU đích; prepare để tải online
PROJECT_ROOT_OVERRIDE = None
BANK_ROOT = Path('/kaggle/working/clindistill_research_v3')
RUN_MINI_REHEARSAL = False
MINI_ROOT_OVERRIDE = None

def find(predicate):
    found = set()
    for base in (Path.cwd(), Path('/kaggle/input')):
        if not base.exists(): continue
        for current, dirs, files in os.walk(base):
            p = Path(current)
            if predicate(p, dirs, files): found.add(p.resolve())
            dirs[:] = [d for d in dirs if d not in {'Images','images','.git','.cache','_support','models','wheels','wheelhouse','official_code','_archive'} and not d.lower().startswith('qwen')]
            if len(p.relative_to(base).parts) >= 8: dirs[:] = []
    return sorted(found, key=str)
def one(override, found, name):
    if override: return Path(override).resolve()
    if len(found) != 1: raise RuntimeError(f'{name}: {found}. Set *_OVERRIDE.')
    return found[0]
source = one(PROJECT_ROOT_OVERRIDE, find(lambda p,d,f:
    (p / 'clindistill/research').is_dir() and 'configs' in d and p != (BANK_ROOT / 'source').resolve()), 'Source v5')
spec = importlib.util.spec_from_file_location('research_bank', source / '_support/research_bank.py')
bank = importlib.util.module_from_spec(spec); spec.loader.exec_module(bank)
print(json.dumps(bank.inventory(), indent=2))
# GPU runtime preflight before downloading large assets
runtime_spec = importlib.util.spec_from_file_location('research_hardware', source / 'clindistill/research/runtime.py')
runtime = importlib.util.module_from_spec(runtime_spec); runtime_spec.loader.exec_module(runtime)
print(json.dumps(runtime.hardware(), indent=2))
if MODE == 'prepare' and RUN_MINI_REHEARSAL:
    if MINI_ROOT_OVERRIDE:
        mini = Path(MINI_ROOT_OVERRIDE)
    else:
        full_release = one(None, find(lambda p,d,f: 'release.json' in f and
            not bank.read_json(p / 'release.json').get('debug_subset')), 'Full prepared COde release')
        full_images = one(None, find(lambda p,d,f: (p / 'Images/Photographs').is_dir() and
            'mini_dataset.json' not in f), 'Full COde images')
        mini = BANK_ROOT.parent / 'research_mini_v1'
        mini_spec = importlib.util.spec_from_file_location('mini_builder', source / '_support/kaggle_offline.py')
        mini_builder = importlib.util.module_from_spec(mini_spec); mini_spec.loader.exec_module(mini_builder)
        if not (mini / 'mini_dataset.json').exists():
            import uuid
            staging = mini.with_name(mini.name + '_staging_' + uuid.uuid4().hex[:8])
            mini_builder.make_mini_code(full_release, full_images, staging, per_class=2, independent_groups=True)
            staging.rename(mini)
        elif bank.read_json(mini / 'prepared/release.json')['source_release_id'] != bank.read_json(full_release / 'release.json')['release_id']:
            raise RuntimeError('Mini thuộc release khác. Đặt MINI_ROOT_OVERRIDE tới mini đúng release.')
# Online preparation helpers (safe with the existing 0.5.1 addon)
def require_environment(expected, actual):
    """CPU online preparation must match the GPU image; never silently replace CUDA."""
    differences = []
    for key in ('python_minor', 'system', 'machine'):
        if expected.get(key) != actual.get(key):
            differences.append(f'{key}: target={expected.get(key)!r}, current={actual.get(key)!r}')
    for key in sorted(set(expected.get('packages', {})) | set(actual.get('packages', {}))):
        wanted, current = expected.get('packages', {}).get(key), actual.get('packages', {}).get(key)
        if wanted != current:
            differences.append(f'{key}: target={wanted!r}, current={current!r}')
    if differences:
        raise RuntimeError('Python/platform/base scientific stack differs from the target GPU image. '
                           'Prepare on a matching online image; do not use this bank on a different stack.\n'
                           + '\n'.join(differences))

def require_preparation_environment(target, actual):
    """Allow only CPU -> CUDA wheel variants of the same Torch/vision releases."""
    import re
    checked = {**actual, 'packages': dict(actual.get('packages', {}))}
    variants = []
    for key in ('torch', 'torchvision'):
        wanted = target.get('packages', {}).get(key, '')
        current = actual.get('packages', {}).get(key, '')
        wanted_release, _, wanted_build = wanted.partition('+')
        current_release, _, current_build = current.partition('+')
        variants.append((key, wanted_release == current_release, wanted_build, current_build))
    bridge = (all(same and cpu == 'cpu' and re.fullmatch(r'cu\d+', cuda)
                  for _, same, cuda, cpu in variants)
              and variants[0][2] == variants[1][2])
    if bridge:
        for key, *_ in variants:
            checked['packages'][key] = target['packages'][key]
    require_environment(target, checked)
    return bool(bridge)

def check_online(opener=None):
    """Public, unauthenticated request: network failure is not a rejected HF token."""
    from urllib.request import Request, urlopen
    from urllib.error import HTTPError
    request = Request('https://huggingface.co', method='HEAD')
    try:
        with (opener or urlopen)(request, timeout=10):
            pass
    except HTTPError as exc:
        status = exc.code
        exc.close()
        raise RuntimeError(f'Online setup: Hugging Face returned HTTP {status} to an unauthenticated request. '
                           'HF_TOKEN has not been read or checked yet. Retry when the service is reachable.') from None
    except (OSError, TimeoutError) as exc:
        raise RuntimeError(f'Online setup cannot reach huggingface.co ({type(exc).__name__}). '
                           'HF_TOKEN has not been read or checked yet. If Kaggle blocks Internet with RTX PRO 6000, '
                           'stop the session, select Accelerator=None (CPU), enable Internet, and restart notebook00. '
                           'Use Pro 6000 offline for notebooks01/02 after saving the bank.') from None

def read_hf_token(secret_reader=None):
    """Read a secret without logging its value or an exception that may contain it."""
    token = os.environ.get('HF_TOKEN')
    if not token or not token.strip():
        try:
            if secret_reader is None:
                from kaggle_secrets import UserSecretsClient
                secret_reader = UserSecretsClient().get_secret
            token = secret_reader('HF_TOKEN')
        except Exception as exc:
            kind = type(exc).__name__
            if isinstance(exc, (ConnectionError, OSError, TimeoutError)):
                hint = 'Kaggle Secrets service is unreachable. Check Internet/session connectivity; this does not mean the HF token is invalid.'
            elif kind == 'CredentialError':
                hint = 'Kaggle denied access to Secrets. Grant this notebook access to HF_TOKEN and restart the session; HF has not validated the token.'
            else:
                hint = 'Cannot retrieve Kaggle Secret HF_TOKEN. Check its exact name and notebook access; HF has not validated the token.'
            raise RuntimeError(f'{hint} Error type: {kind}.') from None
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError('HF_TOKEN is empty. Set a nonempty Kaggle Secret and grant notebook access.')
    os.environ['HF_TOKEN'] = token.strip()

if MODE == 'prepare':
    if TARGET_ENVIRONMENT is None and bank.inventory()['packages']['torch'].endswith('+cpu'):
        raise RuntimeError('CPU online setup needs TARGET_ENVIRONMENT copied from GPU MODE=inspect. '
                           'The bank must record the GPU target, not the CPU Torch build.')
    if TARGET_ENVIRONMENT is not None:
        bridge = require_preparation_environment(TARGET_ENVIRONMENT, bank.inventory())
        if bridge:
            print('CPU online -> CUDA target: same Torch/vision releases; all other packages match. GPU activation remains strict.')
    check_online()
    read_hf_token()
    # Backward-compatible prepare implementation for the existing uploaded 0.5.1 addon.
    exec('def require_environment(expected, actual):\n    """CPU online preparation must match the GPU image; never silently replace CUDA."""\n    differences = []\n    for key in (\'python_minor\', \'system\', \'machine\'):\n        if expected.get(key) != actual.get(key):\n            differences.append(f\'{key}: target={expected.get(key)!r}, current={actual.get(key)!r}\')\n    for key in sorted(set(expected.get(\'packages\', {})) | set(actual.get(\'packages\', {}))):\n        wanted, current = expected.get(\'packages\', {}).get(key), actual.get(\'packages\', {}).get(key)\n        if wanted != current:\n            differences.append(f\'{key}: target={wanted!r}, current={current!r}\')\n    if differences:\n        raise RuntimeError(\'Python/platform/base scientific stack differs from the target GPU image. \'\n                           \'Prepare on a matching online image; do not use this bank on a different stack.\\n\'\n                           + \'\\n\'.join(differences))\n\ndef require_preparation_environment(target, actual):\n    """Allow only CPU -> CUDA wheel variants of the same Torch/vision releases."""\n    import re\n    checked = {**actual, \'packages\': dict(actual.get(\'packages\', {}))}\n    variants = []\n    for key in (\'torch\', \'torchvision\'):\n        wanted = target.get(\'packages\', {}).get(key, \'\')\n        current = actual.get(\'packages\', {}).get(key, \'\')\n        wanted_release, _, wanted_build = wanted.partition(\'+\')\n        current_release, _, current_build = current.partition(\'+\')\n        variants.append((key, wanted_release == current_release, wanted_build, current_build))\n    bridge = (all(same and cpu == \'cpu\' and re.fullmatch(r\'cu\\d+\', cuda)\n                  for _, same, cuda, cpu in variants)\n              and variants[0][2] == variants[1][2])\n    if bridge:\n        for key, *_ in variants:\n            checked[\'packages\'][key] = target[\'packages\'][key]\n    require_environment(target, checked)\n    return bool(bridge)\n\ndef prepare(source, destination, *, target_environment=None):\n    source, destination = Path(source).resolve(), Path(destination).resolve()\n    if platform.system() != \'Linux\':\n        raise RuntimeError(\'Prepare binary wheels on Linux/Kaggle, matching the target Python environment.\')\n    preparation = inventory()\n    base = target_environment if target_environment is not None else preparation\n    cpu_bridge = require_preparation_environment(base, preparation)\n    destination.mkdir(parents=True, exist_ok=True)\n    manifest_path = destination / \'research_bank_manifest.json\'\n    if manifest_path.exists():\n        existing = verify(destination, check_environment=False)\n        require_environment(base, existing[\'base_environment\'])\n        expected_source = {p.relative_to(source).as_posix(): digest(p) for p in source_files(source)}\n        saved = {p.removeprefix(\'source/\'): h for p, h in read_json(manifest_path)[\'files\'].items()\n                 if p.startswith(\'source/\')}\n        if expected_source != saved:\n            raise ValueError(\'Prepared bank belongs to different source. Choose a new bank directory.\')\n        return read_json(manifest_path)\n    constraints = destination / \'base-constraints.txt\'\n    constraints.write_text(\'\'.join(f\'{k}=={v}\\n\' for k, v in base[\'packages\'].items()), encoding=\'utf-8\')\n    wheelhouse = destination / \'wheels\'; wheelhouse.mkdir(exist_ok=True)\n    online_env = os.environ.copy()\n    for key in (\'PIP_NO_INDEX\', \'HF_HUB_OFFLINE\', \'TRANSFORMERS_OFFLINE\'):\n        online_env.pop(key, None)\n    run([sys.executable, \'-m\', \'pip\', \'download\', \'--only-binary=:all:\', \'--constraint\', constraints,\n         \'--dest\', wheelhouse, *ADDONS, *BUILD_TOOLS], env=online_env)\n    # Target Torch/CUDA is supplied by Kaggle; addons must not replace the CPU preparer\'s\n    # Torch or the GPU runtime. No compiled Torch extension is supported by this bridge.\n    forbidden = [p.name for p in wheelhouse.glob(\'*.whl\')\n                 if p.name.split(\'-\')[0].replace(\'_\', \'-\').lower() in {\'torch\', \'torchvision\', \'torchaudio\', \'triton\'}\n                 or p.name.lower().startswith((\'nvidia-\', \'nvidia_\'))]\n    if forbidden:\n        raise ValueError(\'Research addon wheelhouse must not contain Torch/CUDA wheels or extensions: \' + \', \'.join(forbidden))\n    # Freeze the complete downloaded dependency closure, not just the three top-level addons.\n    locked = sorted({f\'{p.name.split("-")[0].replace("_", "-")}=={p.name.split("-")[1]}\'\n                     for p in wheelhouse.glob(\'*.whl\') if not p.name.startswith(\'clindistill_code-\')})\n    lock = destination / \'requirements.lock\'\n    lock.write_text(\'\\n\'.join(locked) + \'\\n\', encoding=\'utf-8\')\n    run([sys.executable, \'-m\', \'pip\', \'install\', \'--no-index\', \'--find-links\', wheelhouse,\n         \'--constraint\', constraints, \'--requirement\', lock])\n    payload = destination / \'source\'\n    for path in source_files(source):\n        target = payload / path.relative_to(source); target.parent.mkdir(parents=True, exist_ok=True)\n        if target.exists() and digest(target) != digest(path):\n            raise ValueError(\'Partially built bank contains different source. Choose a fresh bank name.\')\n        shutil.copy2(path, target)\n    # A fresh build staging folder avoids stale egg-info and duplicate project versions.\n    staging = destination / (\'_build_\' + uuid.uuid4().hex[:8]); staging.mkdir()\n    shutil.copy2(payload / \'pyproject.toml\', staging / \'pyproject.toml\')\n    shutil.copytree(payload / \'clindistill\', staging / \'clindistill\')\n    run([sys.executable, \'-m\', \'pip\', \'wheel\', \'--no-index\', \'--no-deps\', \'--no-build-isolation\',\n         \'--wheel-dir\', wheelhouse, staging])\n    project = list(wheelhouse.glob(\'clindistill_code-*-py3-none-any.whl\'))\n    if len(project) != 1:\n        raise ValueError(\'Expected exactly one fresh project wheel.\')\n    project_files = verify_project_wheel(project[0], payload)\n    run([sys.executable, \'-m\', \'pip\', \'install\', \'--no-index\', \'--no-deps\', \'--force-reinstall\', project[0]])\n    # Do not download gated weights until explicit access has been granted by the model owner.\n    from huggingface_hub import HfApi, snapshot_download\n    token = os.environ.get(\'HF_TOKEN\') or None\n    for key in (\'HF_HUB_OFFLINE\', \'TRANSFORMERS_OFFLINE\'):\n        os.environ.pop(key, None)\n    api = HfApi(token=token); revisions, models = {}, {}\n    revision_path = destination / \'model_revisions.json\'\n    pinned = read_json(revision_path) if revision_path.exists() else {}\n    for role, repo in MODEL_IDS.items():\n        if role in pinned and pinned[role][\'repo\'] != repo:\n            raise ValueError(\'Bank model ID changed. Choose a new bank directory.\')\n        revision = pinned[role][\'revision\'] if role in pinned else api.model_info(repo).sha\n        pinned[role] = {\'repo\': repo, \'revision\': revision}\n        write_json(revision_path, pinned)\n        target = destination / \'models\' / role\n        try:\n            snapshot_download(repo, revision=revision, token=token, local_dir=target,\n                allow_patterns=[\'*.json\', \'*.safetensors\', \'pytorch_model*.bin\', \'*.txt\', \'*.model\'],\n                ignore_patterns=[\'onnx/*\', \'onnx/**\', \'1_Pooling/*\', \'imgs/*\'])\n        except Exception as exc:\n            # Avoid echoing an authenticated request or token in a notebook traceback.\n            raise RuntimeError(f\'Cannot download {repo}. Accept its access terms and set private HF_TOKEN; \'\n                               f\'original error type: {type(exc).__name__}.\') from None\n        revisions[role] = {\'repo\': repo, \'revision\': revision}\n        models[role] = target.relative_to(destination).as_posix()\n    import torch\n    for role, url in CNN_URLS.items():\n        target = destination / \'models\' / Path(url).name; target.parent.mkdir(exist_ok=True)\n        if not target.exists():\n            torch.hub.download_url_to_file(url, target, hash_prefix=Path(url).stem.split(\'-\')[-1])\n        if not digest(target).startswith(Path(url).stem.split(\'-\')[-1]):\n            raise ValueError(\'Official CNN checkpoint checksum mismatch.\')\n        models[role] = target.relative_to(destination).as_posix()\n    # Instantiate each backend locally before marking the bank complete (no network fallback).\n    from transformers import AutoModel, AutoTokenizer\n    for role in MODEL_IDS:\n        folder = destination / models[role]\n        encoder = AutoModel.from_pretrained(folder, local_files_only=True, trust_remote_code=False)\n        expected = {\'student\': (\'dinov3_vit\', 768), \'teacher\': (\'dinov3_vit\', 1024), \'clinical\': (\'xlm-roberta\', 1024)}[role]\n        if (encoder.config.model_type, encoder.config.hidden_size) != expected:\n            raise ValueError(f\'Unexpected {role} architecture.\')\n        if role == \'clinical\':\n            AutoTokenizer.from_pretrained(folder, local_files_only=True, use_fast=True)\n        del encoder\n    require_preparation_environment(base, inventory())\n    files = {p.relative_to(destination).as_posix(): digest(p)\n             for folder in (\'source\', \'models\', \'wheels\') for p in sorted((destination / folder).rglob(\'*\'))\n             if p.is_file() and not {\'.cache\', \'__pycache__\'} & set(p.relative_to(destination).parts)}\n    files[\'base-constraints.txt\'] = digest(constraints)\n    files[\'model_revisions.json\'] = digest(revision_path)\n    files[\'requirements.lock\'] = digest(lock)\n    manifest = {\'schema_version\': BANK_SCHEMA, \'status\': \'assets_verified_backends_loaded\', \'base_environment\': base,\n                \'preparation_environment\': preparation, \'cpu_to_cuda_preparation\': cpu_bridge,\n                \'backend_validation\': \'Local CPU model loading only; GPU execution and full experiments remain unverified.\',\n                \'models\': models, \'model_revisions\': revisions, \'files\': files,\n                \'project_wheel\': project[0].relative_to(destination).as_posix(), \'project_files\': project_files,\n                \'source_files\': sorted(p for p in files if p.startswith(\'source/\')),\n                \'note\': \'Backend loading passed; run the small end-to-end research rehearsal before a full study.\'}\n    write_json(manifest_path, manifest)\n    print(\'BANK READY:\', destination, flush=True)\n    return manifest', bank.__dict__)
    info = bank.prepare(source, BANK_ROOT, target_environment=TARGET_ENVIRONMENT)
elif MODE == 'inspect':
    print('INSPECT ONLY: no network, Secrets or downloads. Copy the inventory JSON into TARGET_ENVIRONMENT on the online CPU notebook.')
else:
    raise ValueError('MODE must be inspect or prepare.')


## Rehearsal tùy chọn sau khi có prepared release

Lần đầu chỉ tạo bank. Sau notebook01, có thể đặt RUN_MINI_REHEARSAL=True,
gắn prepared release và ảnh để thử pretrained thật trên mini độc lập theo nhóm bệnh nhân.
Đây là kiểm tra phần mềm, không phải kết quả nghiên cứu; không mở test.


In [ ]:
if MODE == 'prepare' and RUN_MINI_REHEARSAL:
    bank.installed_project_matches(info['project_files'])
    from clindistill.artifacts import read_json, write_json
    from clindistill.research.data import prepare_features
    from clindistill.research.training import run_study
    from clindistill.research.training import grouped_folds
    rows = [json.loads(line) for line in (mini / 'prepared/student_single_train.jsonl').read_text(encoding='utf-8').splitlines() if line]
    grouped_folds([r['single_label_index'] for r in rows], [r['split_group'] for r in rows], 2, 2026)
    models = {k: BANK_ROOT / v for k,v in info['models'].items()}
    cfg = read_json(BANK_ROOT / 'source/configs/code/research.json')
    cfg.update(debug=True, seeds=[2026], folds=2, epochs=3, patience=3, feature_dim=64,
        learning_rate=0.01, bootstrap_repetitions=10,
        methods=['student_no_kd','kd_visual','kd_multimodal','kd_confidence','kd_reliability'])
    cache = BANK_ROOT.parent / 'research_mini_features'
    with bank.block_network():
        prepare_features(mini / 'prepared', mini, models, cache, cfg)
        display(run_study(cache, BANK_ROOT.parent / 'research_mini_runs', cfg, require_gpu=False))
    write_json(BANK_ROOT / 'rehearsal.json', {'status':'passed','debug':True,'contains_test':False})
print('Save Version. 01/02/03 Add Input > Notebook Output > chọn output có research_bank_manifest.json.')


Bank gồm source, Linux wheels, DINO-B/L, BGE-M3, ResNet/Swin weights; không gồm COde đầy đủ.
Giữ bank, mini, cache và checkpoints riêng tư. Nếu chưa có quyền DINO, dừng ở tải model và hoàn tất quyền truy cập;
không có cơ chế thay model hoặc sinh kết quả giả. Resume prepare dùng lại model downloads đã có.